## Imports

In [17]:
import pandas as pd
import numpy as np
import scipy
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

### Leitura do CSV baixado do Kaggle

In [18]:
caminho_arquivo = "AI_Impact_Student_Life_2026.csv"

df_completo = pd.read_csv(
    caminho_arquivo,
    sep=",",
    encoding="utf-8",
    decimal="."
)

df_completo.columns = df_completo.columns.str.strip()

### Seleção apenas das colunas de interesse

In [19]:
colunas_interesse = [
    "Student_ID",
    "Task_Frequency_Daily",
    "Main_Usage_Case",
    "GPA_Baseline",
    "GPA_Post_AI"
]

df = df_completo[colunas_interesse].copy()

### Inspeção inicial

In [ ]:
print("Formato do DataFrame (linhas, colunas):", df.shape)
print("\nPrimeiras linhas:")
print(df.head())

print("\nTipos de dados por coluna:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

print("\nResumo estatístico (colunas numéricas):")
print(df.describe())

# ---- Identificação de dados duplicados ----

# Duplicatas exatas (todas as colunas iguais)
qtd_duplicadas_exatas = df.duplicated().sum()
print(f"\nQuantidade de linhas totalmente duplicadas: {qtd_duplicadas_exatas}")

# Duplicatas por Student_ID (mesmo aluno aparecendo mais de uma vez)
qtd_ids_duplicados = df.duplicated(subset="Student_ID").sum()
print(f"Quantidade de Student_ID duplicados: {qtd_ids_duplicados}")

if qtd_ids_duplicados > 0:
    print("\nExemplos de registros com Student_ID duplicado:")
    print(df[df.duplicated(subset="Student_ID", keep=False)].sort_values(by="Student_ID").head(10))

# ---- Tratamento: remoção das duplicatas por Student_ID ----
df = df.drop_duplicates(subset="Student_ID", keep="first")
print(f"\nFormato do DataFrame após remoção de duplicatas por Student_ID: {df.shape}")

### Ajustes de formatação

In [21]:
# Garante que as colunas de GPA são numéricas
df["GPA_Baseline"] = pd.to_numeric(df["GPA_Baseline"], errors="coerce")
df["GPA_Post_AI"] = pd.to_numeric(df["GPA_Post_AI"], errors="coerce")

# Garante que a frequência de uso também é numérica
df["Task_Frequency_Daily"] = pd.to_numeric(df["Task_Frequency_Daily"], errors="coerce")

# Remove linhas com dados faltando nas colunas essenciais
df = df.dropna(subset=["GPA_Baseline", "GPA_Post_AI"])

# Remove linhas totalmente vazias
df = df.dropna(how="all")

### Coluna calculada: variação de GPA

In [22]:
df["GPA_Variacao"] = df["GPA_Post_AI"] - df["GPA_Baseline"]

### Correlação de Pearson: Task_Frequency_Daily (Frequência diária do uso de agentes IA) x GPA_Variacao

In [ ]:
coef_pearson, p_valor_pearson = stats.pearsonr(df["Task_Frequency_Daily"], df["GPA_Variacao"])

print(f"\nCoeficiente de correlação de Pearson: {coef_pearson:.4f}")
print(f"P-valor (Pearson): {p_valor_pearson:.5f}")

### Correlação de Spearman: Task_Frequency_Daily (Frequência diária do uso de agentes IA) x GPA_Variacao

In [ ]:
coef_spearman, p_valor_spearman = stats.spearmanr(df["Task_Frequency_Daily"], df["GPA_Variacao"])

print(f"\nCoeficiente de correlação de Spearman: {coef_spearman:.4f}")
print(f"P-valor (Spearman): {p_valor_spearman:.5f}")

### Interpretação da força da correlação

In [ ]:
def interpretar_correlacao(r):
    r_abs = abs(r)
    if r_abs < 0.1:
        return "praticamente nenhuma correlação"
    elif r_abs < 0.3:
        return "correlação fraca"
    elif r_abs < 0.5:
        return "correlação moderada"
    elif r_abs < 0.7:
        return "correlação forte"
    else:
        return "correlação muito forte"

def resumo_correlacao(nome, r, p):
    direcao = "positiva" if r > 0 else "negativa"
    forca = interpretar_correlacao(r)
    significativo = "estatisticamente significativa" if p < 0.05 else "não estatisticamente significativa (p >= 0.05)"
    print(f"\n[{nome}] correlação {forca} e {direcao} (r = {r:.4f}). Resultado {significativo} (p = {p:.5f}).")

resumo_correlacao("Pearson", coef_pearson, p_valor_pearson)
resumo_correlacao("Spearman", coef_spearman, p_valor_spearman)